# Synthetic-to-Real Credit Card Fraud Transfer
## Dataset Reconnaissance Notebook

### Objective

This notebook performs a **schema-first audit** of the two mounted datasets:

1. **IBM Credit Card Transactions**
   - `credit_card_transactions-ibm_v2.csv`
   - `User0_credit_card_transactions.csv`
   - `sd254_cards.csv`
   - `sd254_users.csv`

2. **ULB / Worldline Credit Card Fraud Detection**
   - `creditcard.csv`

The notebook is intentionally **non-destructive**:
- no rows are modified
- no preprocessing is applied to the source files
- no resampling is performed
- no model is trained
- the large IBM transaction file is inspected in chunks rather than loaded entirely into RAM

### Research goal

We are investigating whether the IBM transaction environment can support a defensible **synthetic-to-real fraud-transfer study** using the real ULB/Worldline dataset as external validation.

The first task is to determine the exact schemas, entity identifiers, temporal fields, fraud labels, and behavioral information available in the mounted IBM files.

In [1]:
# ============================================================
# 1. Imports and environment
# ============================================================

import os
import re
import json
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display, Markdown

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 180)

print("Python environment ready.")

Python environment ready.


## 2. Locate the mounted datasets

The notebook is designed primarily for **Kaggle Notebook** paths (`/kaggle/input`), but it also searches `/mnt/data` and the current working directory.

Because Kaggle may create dataset-folder names that differ from the visible dataset title, the notebook searches by **filename pattern** instead of hard-coding a single folder name.

In [2]:
# ============================================================
# 2. Discover mounted CSV files
# ============================================================

SEARCH_ROOTS = [
    Path("/kaggle/input"),
    Path("/mnt/data"),
    Path(".")
]

csv_files = []

for root in SEARCH_ROOTS:
    if root.exists():
        try:
            csv_files.extend([p for p in root.rglob("*.csv") if p.is_file()])
        except Exception as e:
            print(f"Could not fully scan {root}: {e}")

# Remove duplicates while preserving order
seen = set()
csv_files_unique = []
for p in csv_files:
    rp = str(p.resolve())
    if rp not in seen:
        seen.add(rp)
        csv_files_unique.append(p)

print(f"CSV files discovered: {len(csv_files_unique)}\n")

for p in csv_files_unique:
    size_mb = p.stat().st_size / (1024**2)
    print(f"{size_mb:10.2f} MB  |  {p}")

CSV files discovered: 5

      0.21 MB  |  /kaggle/input/datasets/ealtman2019/credit-card-transactions/sd254_users.csv
      1.81 MB  |  /kaggle/input/datasets/ealtman2019/credit-card-transactions/User0_credit_card_transactions.csv
      0.46 MB  |  /kaggle/input/datasets/ealtman2019/credit-card-transactions/sd254_cards.csv
   2241.84 MB  |  /kaggle/input/datasets/ealtman2019/credit-card-transactions/credit_card_transactions-ibm_v2.csv
    143.84 MB  |  /kaggle/input/datasets/mlg-ulb/creditcardfraud/creditcard.csv


In [3]:
# ============================================================
# 3. Automatically identify the expected files
# ============================================================

def find_by_name(patterns, files):
    matches = []
    for p in files:
        name = p.name.lower()
        if any(pattern.lower() in name for pattern in patterns):
            matches.append(p)
    return matches

ulb_candidates = find_by_name(
    ["creditcard.csv"],
    csv_files_unique
)

ibm_main_candidates = find_by_name(
    ["credit_card_transactions-ibm_v2.csv"],
    csv_files_unique
)

ibm_user0_candidates = find_by_name(
    ["user0_credit_card_transactions.csv"],
    csv_files_unique
)

ibm_cards_candidates = find_by_name(
    ["sd254_cards.csv"],
    csv_files_unique
)

ibm_users_candidates = find_by_name(
    ["sd254_users.csv"],
    csv_files_unique
)

print("ULB candidates:")
for p in ulb_candidates:
    print(" ", p)

print("\nIBM main transaction candidates:")
for p in ibm_main_candidates:
    print(" ", p)

print("\nIBM User0 candidates:")
for p in ibm_user0_candidates:
    print(" ", p)

print("\nIBM cards candidates:")
for p in ibm_cards_candidates:
    print(" ", p)

print("\nIBM users candidates:")
for p in ibm_users_candidates:
    print(" ", p)

if not ulb_candidates:
    raise FileNotFoundError("Could not find creditcard.csv.")

if not ibm_main_candidates:
    raise FileNotFoundError(
        "Could not find credit_card_transactions-ibm_v2.csv. "
        "Run the discovery cell and check the mounted filename."
    )

ULB_PATH = str(ulb_candidates[0])
IBM_MAIN_PATH = str(ibm_main_candidates[0])

IBM_USER0_PATH = str(ibm_user0_candidates[0]) if ibm_user0_candidates else None
IBM_CARDS_PATH = str(ibm_cards_candidates[0]) if ibm_cards_candidates else None
IBM_USERS_PATH = str(ibm_users_candidates[0]) if ibm_users_candidates else None

print("\nSelected files:")
print("ULB       :", ULB_PATH)
print("IBM main  :", IBM_MAIN_PATH)
print("IBM User0 :", IBM_USER0_PATH)
print("IBM cards :", IBM_CARDS_PATH)
print("IBM users :", IBM_USERS_PATH)

ULB candidates:
  /kaggle/input/datasets/mlg-ulb/creditcardfraud/creditcard.csv

IBM main transaction candidates:
  /kaggle/input/datasets/ealtman2019/credit-card-transactions/credit_card_transactions-ibm_v2.csv

IBM User0 candidates:
  /kaggle/input/datasets/ealtman2019/credit-card-transactions/User0_credit_card_transactions.csv

IBM cards candidates:
  /kaggle/input/datasets/ealtman2019/credit-card-transactions/sd254_cards.csv

IBM users candidates:
  /kaggle/input/datasets/ealtman2019/credit-card-transactions/sd254_users.csv

Selected files:
ULB       : /kaggle/input/datasets/mlg-ulb/creditcardfraud/creditcard.csv
IBM main  : /kaggle/input/datasets/ealtman2019/credit-card-transactions/credit_card_transactions-ibm_v2.csv
IBM User0 : /kaggle/input/datasets/ealtman2019/credit-card-transactions/User0_credit_card_transactions.csv
IBM cards : /kaggle/input/datasets/ealtman2019/credit-card-transactions/sd254_cards.csv
IBM users : /kaggle/input/datasets/ealtman2019/credit-card-transactions/

## 4. File sizes

This is important because the IBM transaction file may be very large. We will avoid `pd.read_csv()` on the entire file during reconnaissance.

In [4]:
# ============================================================
# 4. File sizes
# ============================================================

def file_info(path):
    if path is None:
        return None
    p = Path(path)
    size_gb = p.stat().st_size / (1024**3)
    size_mb = p.stat().st_size / (1024**2)
    return {
        "file": p.name,
        "size_MB": round(size_mb, 2),
        "size_GB": round(size_gb, 3),
        "path": str(p)
    }

files_to_report = [
    ULB_PATH,
    IBM_MAIN_PATH,
    IBM_USER0_PATH,
    IBM_CARDS_PATH,
    IBM_USERS_PATH
]

display(pd.DataFrame([file_info(p) for p in files_to_report if p]))

,file,size_MB,size_GB,path
0,creditcard.csv,143.84,0.140,/kaggle/input/datasets/mlg-ulb/creditcardfraud...
1,credit_card_transactions-ibm_v2.csv,2241.84,2.189,/kaggle/input/datasets/ealtman2019/credit-card...
2,User0_credit_card_transactions.csv,1.81,0.002,/kaggle/input/datasets/ealtman2019/credit-card...
3,sd254_cards.csv,0.46,0.000,/kaggle/input/datasets/ealtman2019/credit-card...
4,sd254_users.csv,0.21,0.000,/kaggle/input/datasets/ealtman2019/credit-card...


## 5. Load small samples

The IBM main file is read using `nrows` only.

The purpose here is to discover:
- column names
- dtypes
- example values
- likely identifiers
- target/fraud fields
- time/date fields

In [5]:
# ============================================================
# 5. Sample loading
# ============================================================

SAMPLE_ROWS = 10_000

ibm_sample = pd.read_csv(
    IBM_MAIN_PATH,
    nrows=SAMPLE_ROWS,
    low_memory=False
)

ulb_sample = pd.read_csv(
    ULB_PATH,
    nrows=min(SAMPLE_ROWS, 500_000),
    low_memory=False
)

print("IBM sample shape:", ibm_sample.shape)
print("ULB sample shape:", ulb_sample.shape)

if IBM_USER0_PATH:
    ibm_user0_sample = pd.read_csv(
        IBM_USER0_PATH,
        nrows=10_000,
        low_memory=False
    )
else:
    ibm_user0_sample = None

if IBM_CARDS_PATH:
    ibm_cards_sample = pd.read_csv(
        IBM_CARDS_PATH,
        nrows=10_000,
        low_memory=False
    )
else:
    ibm_cards_sample = None

if IBM_USERS_PATH:
    ibm_users_sample = pd.read_csv(
        IBM_USERS_PATH,
        nrows=10_000,
        low_memory=False
    )
else:
    ibm_users_sample = None

IBM sample shape: (10000, 15)
ULB sample shape: (10000, 31)


In [6]:
# ============================================================
# 6. First rows and column lists
# ============================================================

print("=" * 90)
print("IBM MAIN TRANSACTION FILE")
print("=" * 90)
display(ibm_sample.head())

print("\nIBM columns:")
for i, col in enumerate(ibm_sample.columns):
    print(f"{i:3d}: {col}")

print("\n" + "=" * 90)
print("ULB CREDITCARD FILE")
print("=" * 90)
display(ulb_sample.head())

print("\nULB columns:")
for i, col in enumerate(ulb_sample.columns):
    print(f"{i:3d}: {col}")

IBM MAIN TRANSACTION FILE


,User,Card,Year,Month,Day,Time,Amount,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Errors?,Is Fraud?
0,0,0,2002,9,1,06:21,$134.09,Swipe Transaction,3527213246127876953,La Verne,CA,91750.0,5300,NaN,No
1,0,0,2002,9,1,06:42,$38.48,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No
2,0,0,2002,9,2,06:22,$120.34,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No
3,0,0,2002,9,2,17:45,$128.95,Swipe Transaction,3414527459579106770,Monterey Park,CA,91754.0,5651,NaN,No
4,0,0,2002,9,3,06:23,$104.71,Swipe Transaction,5817218446178736267,La Verne,CA,91750.0,5912,NaN,No



IBM columns:
  0: User
  1: Card
  2: Year
  3: Month
  4: Day
  5: Time
  6: Amount
  7: Use Chip
  8: Merchant Name
  9: Merchant City
 10: Merchant State
 11: Zip
 12: MCC
 13: Errors?
 14: Is Fraud?

ULB CREDITCARD FILE


,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,V11,V12,V13,V14,V15,V16,V17,V18,V19,V20,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,0.090794,-0.551600,-0.617801,-0.991390,-0.311169,1.468177,-0.470401,0.207971,0.025791,0.403993,0.251412,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,-0.166974,1.612727,1.065235,0.489095,-0.143772,0.635558,0.463917,-0.114805,-0.183361,-0.145783,-0.069083,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,0.207643,0.624501,0.066084,0.717293,-0.165946,2.345865,-2.890083,1.109969,-0.121359,-2.261857,0.524980,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,-0.054952,-0.226487,0.178228,0.507757,-0.287924,-0.631418,-1.059647,-0.684093,1.965775,-1.232622,-0.208038,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,0.753074,-0.822843,0.538196,1.345852,-1.119670,0.175121,-0.451449,-0.237033,-0.038195,0.803487,0.408542,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0



ULB columns:
  0: Time
  1: V1
  2: V2
  3: V3
  4: V4
  5: V5
  6: V6
  7: V7
  8: V8
  9: V9
 10: V10
 11: V11
 12: V12
 13: V13
 14: V14
 15: V15
 16: V16
 17: V17
 18: V18
 19: V19
 20: V20
 21: V21
 22: V22
 23: V23
 24: V24
 25: V25
 26: V26
 27: V27
 28: V28
 29: Amount
 30: Class


## 7. Inspect all IBM auxiliary files

The IBM dataset shown in the mounted environment contains four useful components:

- main transaction table
- User0 transaction sample
- card table
- user table

We need to understand their relationships before deciding whether the main transaction table can be used alone or whether the card/user tables should be joined.

In [7]:
# ============================================================
# 7. Auxiliary IBM schemas
# ============================================================

def show_schema(df, name):
    if df is None:
        print(f"{name}: NOT FOUND")
        return

    print("=" * 90)
    print(name)
    print("=" * 90)
    print("Shape:", df.shape)
    print("\nColumns:")
    for i, col in enumerate(df.columns):
        print(f"{i:3d}: {col}")

    print("\nDtypes:")
    display(df.dtypes.to_frame("dtype"))

    print("\nFirst 5 rows:")
    display(df.head())


show_schema(ibm_user0_sample, "IBM User0 Transactions")
show_schema(ibm_cards_sample, "IBM Cards")
show_schema(ibm_users_sample, "IBM Users")

IBM User0 Transactions
Shape: (10000, 15)

Columns:
  0: User
  1: Card
  2: Year
  3: Month
  4: Day
  5: Time
  6: Amount
  7: Use Chip
  8: Merchant Name
  9: Merchant City
 10: Merchant State
 11: Zip
 12: MCC
 13: Errors?
 14: Is Fraud?

Dtypes:


,dtype
User,int64
Card,int64
Year,int64
Month,int64
Day,int64
Time,object
Amount,object
Use Chip,object
Merchant Name,int64
Merchant City,object



First 5 rows:


,User,Card,Year,Month,Day,Time,Amount,Use Chip,Merchant Name,Merchant City,Merchant State,Zip,MCC,Errors?,Is Fraud?
0,0,0,2002,9,1,06:21,$134.09,Swipe Transaction,3527213246127876953,La Verne,CA,91750.0,5300,NaN,No
1,0,0,2002,9,1,06:42,$38.48,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No
2,0,0,2002,9,2,06:22,$120.34,Swipe Transaction,-727612092139916043,Monterey Park,CA,91754.0,5411,NaN,No
3,0,0,2002,9,2,17:45,$128.95,Swipe Transaction,3414527459579106770,Monterey Park,CA,91754.0,5651,NaN,No
4,0,0,2002,9,3,06:23,$104.71,Swipe Transaction,5817218446178736267,La Verne,CA,91750.0,5912,NaN,No


IBM Cards
Shape: (6146, 13)

Columns:
  0: User
  1: CARD INDEX
  2: Card Brand
  3: Card Type
  4: Card Number
  5: Expires
  6: CVV
  7: Has Chip
  8: Cards Issued
  9: Credit Limit
 10: Acct Open Date
 11: Year PIN last Changed
 12: Card on Dark Web

Dtypes:


...

,User,CARD INDEX,Card Brand,Card Type,Card Number,Expires,CVV,Has Chip,Cards Issued,Credit Limit,Acct Open Date,Year PIN last Changed,Card on Dark Web
0,0,0,Visa,Debit,4344676511950444,12/2022,623,YES,2,$24295,09/2002,2008,No
1,0,1,Visa,Debit,4956965974959986,12/2020,393,YES,2,$21968,04/2014,2014,No
2,0,2,Visa,Debit,4582313478255491,02/2024,719,YES,2,$46414,07/2003,2004,No
3,0,3,Visa,Credit,4879494103069057,08/2024,693,NO,1,$12400,01/2003,2012,No
4,0,4,Mastercard,Debit (Prepaid),5722874738736011,03/2009,75,YES,1,$28,09/2008,2009,No


IBM Users
Shape: (2000, 18)

Columns:
  0: Person
  1: Current Age
  2: Retirement Age
  3: Birth Year
  4: Birth Month
  5: Gender
  6: Address
  7: Apartment
  8: City
  9: State
 10: Zipcode
 11: Latitude
 12: Longitude
 13: Per Capita Income - Zipcode
 14: Yearly Income - Person
 15: Total Debt
 16: FICO Score
 17: Num Credit Cards

Dtypes:


,dtype
Person,object
Current Age,int64
Retirement Age,int64
Birth Year,int64
Birth Month,int64
Gender,object
Address,object
Apartment,float64
City,object
State,object



First 5 rows:


,Person,Current Age,Retirement Age,Birth Year,Birth Month,Gender,Address,Apartment,City,State,Zipcode,Latitude,Longitude,Per Capita Income - Zipcode,Yearly Income - Person,Total Debt,FICO Score,Num Credit Cards
0,Hazel Robinson,53,66,1966,11,Female,462 Rose Lane,NaN,La Verne,CA,91750,34.15,-117.76,$29278,$59696,$127613,787,5
1,Sasha Sadr,53,68,1966,12,Female,3606 Federal Boulevard,NaN,Little Neck,NY,11363,40.76,-73.74,$37891,$77254,$191349,701,5
2,Saanvi Lee,81,67,1938,11,Female,766 Third Drive,NaN,West Covina,CA,91792,34.02,-117.89,$22681,$33483,$196,698,5
3,Everlee Clark,63,63,1957,1,Female,3 Madison Street,NaN,New York,NY,10069,40.71,-73.99,$163145,$249925,$202328,722,4
4,Kyle Peterson,43,70,1976,9,Male,9620 Valley Stream Drive,NaN,San Francisco,CA,94117,37.76,-122.44,$53797,$109687,$183855,675,1


## 8. Data types and missing values

Missing values matter because a cross-dataset representation should not depend on fields that are sparsely populated.

In [8]:
# ============================================================
# 8. Missing-value reports
# ============================================================

def missing_report(df, name):
    report = pd.DataFrame({
        "column": df.columns,
        "dtype": [str(df[c].dtype) for c in df.columns],
        "missing_count": [int(df[c].isna().sum()) for c in df.columns],
        "missing_pct": [float(df[c].isna().mean() * 100) for c in df.columns],
        "unique_values": [int(df[c].nunique(dropna=True)) for c in df.columns]
    })

    report = report.sort_values(
        ["missing_pct", "unique_values"],
        ascending=[False, False]
    )

    print(f"\n{name}")
    display(report)

    return report


ibm_missing = missing_report(
    ibm_sample,
    "IBM Main Transaction Missingness"
)

ulb_missing = missing_report(
    ulb_sample,
    "ULB Missingness"
)


IBM Main Transaction Missingness


,column,dtype,missing_count,missing_pct,unique_values
13,Errors?,object,9739,97.39,7
11,Zip,float64,748,7.48,283
10,Merchant State,object,584,5.84,37
6,Amount,object,0,0.00,6853
5,Time,object,0,0.00,962
8,Merchant Name,int64,0,0.00,407
9,Merchant City,object,0,0.00,209
12,MCC,int64,0,0.00,81
4,Day,int64,0,0.00,31
2,Year,int64,0,0.00,19



ULB Missingness


,column,dtype,missing_count,missing_pct,unique_values
1,V1,float64,0,0.0,9790
2,V2,float64,0,0.0,9790
3,V3,float64,0,0.0,9790
4,V4,float64,0,0.0,9790
5,V5,float64,0,0.0,9790
6,V6,float64,0,0.0,9790
7,V7,float64,0,0.0,9790
8,V8,float64,0,0.0,9790
9,V9,float64,0,0.0,9790
10,V10,float64,0,0.0,9790


In [9]:
# ============================================================
# 9. Numerical summaries
# ============================================================

def numerical_summary(df, name):
    num_cols = df.select_dtypes(include=np.number).columns.tolist()

    print(f"\n{name} numerical columns:")
    print(num_cols)

    if num_cols:
        summary = df[num_cols].describe(
            percentiles=[
                0.01, 0.05, 0.25, 0.50,
                0.75, 0.95, 0.99
            ]
        ).T

        display(summary)
        return summary

    print("No numerical columns found.")
    return pd.DataFrame()


ibm_numeric_summary = numerical_summary(
    ibm_sample,
    "IBM Main"
)

ulb_numeric_summary = numerical_summary(
    ulb_sample,
    "ULB"
)


IBM Main numerical columns:
['User', 'Card', 'Year', 'Month', 'Day', 'Merchant Name', 'Zip', 'MCC']


,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
User,10000.0,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00
Card,10000.0,8.775000e-01,9.299356e-01,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,2.000000e+00,2.000000e+00,2.000000e+00,2.000000e+00
Year,10000.0,2.010512e+03,4.964707e+00,2.002000e+03,2.002000e+03,2.003000e+03,2.006000e+03,2.011000e+03,2.015000e+03,2.018000e+03,2.019000e+03,2.020000e+03
Month,10000.0,6.625600e+00,3.487440e+00,1.000000e+00,1.000000e+00,1.000000e+00,4.000000e+00,7.000000e+00,1.000000e+01,1.200000e+01,1.200000e+01,1.200000e+01
Day,10000.0,1.579680e+01,8.796434e+00,1.000000e+00,1.000000e+00,2.000000e+00,8.000000e+00,1.600000e+01,2.300000e+01,3.000000e+01,3.100000e+01,3.100000e+01
Merchant Name,10000.0,7.145673e+17,4.033544e+18,-9.092677e+18,-8.920348e+18,-5.904117e+18,-1.288082e+18,9.703280e+16,4.060647e+18,6.661973e+18,8.080935e+18,9.137769e+18
Zip,9252.0,8.890322e+04,1.326266e+04,1.012000e+03,8.093000e+03,7.827000e+04,9.175000e+04,9.175000e+04,9.175200e+04,9.175500e+04,9.430600e+04,9.950400e+04
MCC,10000.0,5.674432e+03,6.946909e+02,1.711000e+03,3.596000e+03,4.829000e+03,5.411000e+03,5.541000e+03,5.912000e+03,7.538000e+03,7.832000e+03,9.402000e+03



ULB numerical columns:
['Time', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28', 'Amount', 'Class']


,count,mean,std,min,1%,5%,25%,50%,75%,95%,99%,max
Time,10000.0,5966.033400,4473.403739,0.000000,68.000000,368.000000,2072.750000,4563.500000,10233.250000,14107.700000,14618.010000,15012.000000
V1,10000.0,-0.241862,1.521679,-27.670569,-5.274072,-2.692177,-1.013283,-0.372799,1.150864,1.318462,1.438628,1.960497
V2,10000.0,0.281949,1.308139,-34.607649,-3.426616,-1.356185,-0.208342,0.288524,0.901879,1.881620,3.713034,8.636214
V3,10000.0,0.906270,1.159154,-15.496222,-2.473296,-0.901840,0.412799,0.944361,1.602903,2.477620,3.105983,4.101716
V4,10000.0,0.264148,1.441235,-4.657545,-3.090398,-2.151396,-0.614424,0.219861,1.125666,2.784392,3.703861,10.463020
V5,10000.0,-0.046398,1.182935,-32.092129,-2.345045,-1.463410,-0.643390,-0.152769,0.371081,1.928648,3.015331,34.099309
V6,10000.0,0.133108,1.307311,-23.496714,-1.927889,-1.247645,-0.629934,-0.152566,0.505357,3.267448,4.117313,21.393069
V7,10000.0,-0.071689,1.077430,-26.548144,-2.738769,-1.251473,-0.542336,-0.055585,0.476280,1.118563,2.056708,34.303177
V8,10000.0,-0.064778,1.259064,-23.632502,-4.303420,-1.010870,-0.190747,0.012865,0.274533,0.909747,1.641247,5.060381
V9,10000.0,0.802224,1.155198,-6.329801,-1.925902,-0.920255,0.070868,0.805275,1.506299,2.522742,3.864147,10.392889


In [10]:
# ============================================================
# 10. Categorical/object columns
# ============================================================

def categorical_summary(df, name, max_display=20):
    cat_cols = df.select_dtypes(
        include=["object", "category"]
    ).columns.tolist()

    print(f"\n{name} categorical/object columns:")
    print(cat_cols)

    for col in cat_cols:
        print(f"\n--- {col} ---")
        print("Unique:", df[col].nunique(dropna=True))

        display(
            df[col]
            .value_counts(dropna=False)
            .head(max_display)
            .to_frame("count")
        )


categorical_summary(
    ibm_sample,
    "IBM Main"
)

categorical_summary(
    ulb_sample,
    "ULB"
)


IBM Main categorical/object columns:
['Time', 'Amount', 'Use Chip', 'Merchant City', 'Merchant State', 'Errors?', 'Is Fraud?']

--- Time ---
Unique: 962


,count
Time,
06:16,90
06:21,88
06:47,87
06:12,87
06:05,87
06:04,86
06:10,85
06:37,84
06:58,84



--- Amount ---
Unique: 6853


,count
Amount,
$140.00,34
$100.00,21
$120.00,19
$80.00,11
$-93.00,11
$93.00,11
$88.00,10
$-100.00,10
$66.00,10



--- Use Chip ---
Unique: 3


...


--- Errors? ---
Unique: 7


,count
Errors?,
NaN,9739
Insufficient Balance,190
Bad PIN,39
Technical Glitch,22
Bad Expiration,5
Bad Card Number,3
"Bad PIN,Insufficient Balance",1
"Bad PIN,Technical Glitch",1



--- Is Fraud? ---
Unique: 2


,count
Is Fraud?,
No,9985
Yes,15



ULB categorical/object columns:
[]


## 11. Search for target/fraud columns

We should not assume that the IBM fraud target has a particular name.

In [11]:
# ============================================================
# 11. Target candidates
# ============================================================

TARGET_KEYWORDS = [
    "fraud",
    "fraudulent",
    "is_fraud",
    "fraud_flag",
    "class",
    "label",
    "target",
    "anomaly"
]

def target_candidates(df):
    results = []

    for col in df.columns:
        low = col.lower().replace(" ", "_")

        if any(k in low for k in TARGET_KEYWORDS):
            results.append(col)

    return results


print("IBM target candidates:")
print(target_candidates(ibm_sample))

print("\nULB target candidates:")
print(target_candidates(ulb_sample))

IBM target candidates:
['Is Fraud?']

ULB target candidates:
['Class']


In [12]:
# ============================================================
# 12. ULB target distribution
# ============================================================

ULB_TARGET = "Class"

print("ULB target distribution:")
display(
    ulb_sample[ULB_TARGET]
    .value_counts(dropna=False)
    .to_frame("count")
)

print("\nULB target percentage:")
display(
    (
        ulb_sample[ULB_TARGET]
        .value_counts(normalize=True, dropna=False) * 100
    ).to_frame("percentage")
)

ULB target distribution:


,count
Class,
0,9962
1,38



ULB target percentage:


,percentage
Class,
0,99.62
1,0.38


### IBM fraud target

The following cell automatically selects a likely IBM target if exactly one candidate is found.

If multiple candidates appear, the notebook stops and asks you to inspect them rather than silently choosing the wrong field.

In [13]:
# ============================================================
# 13. Automatically identify IBM target
# ============================================================

ibm_target_candidates = target_candidates(ibm_sample)

print("Candidates:", ibm_target_candidates)

if len(ibm_target_candidates) == 1:
    IBM_TARGET = ibm_target_candidates[0]
    print("Automatically selected IBM target:", IBM_TARGET)
elif len(ibm_target_candidates) == 0:
    IBM_TARGET = None
    print(
        "No obvious fraud target was detected. "
        "We will inspect the schema manually."
    )
else:
    IBM_TARGET = None
    print(
        "Multiple candidates detected. "
        "Do NOT select one automatically."
    )

if IBM_TARGET is not None:
    print("\nIBM target distribution:")
    display(
        ibm_sample[IBM_TARGET]
        .value_counts(dropna=False)
        .to_frame("count")
    )

    print("\nIBM target percentage:")
    display(
        (
            ibm_sample[IBM_TARGET]
            .value_counts(normalize=True, dropna=False) * 100
        ).to_frame("percentage")
    )

Candidates: ['Is Fraud?']
Automatically selected IBM target: Is Fraud?

IBM target distribution:


,count
Is Fraud?,
No,9985
Yes,15



IBM target percentage:


,percentage
Is Fraud?,
No,99.85
Yes,0.15


## 14. Entity/identifier discovery

This is one of the most important parts of the reconnaissance.

For the IBM dataset, we want to know whether transactions can be grouped by:
- customer/user
- card
- account
- merchant
- transaction

Repeated customer/card histories would allow us to construct behavioral features such as:
- transaction velocity
- amount deviation
- merchant novelty
- time-of-day deviation
- recent transaction frequency

These may become the foundation of the synthetic-to-real transfer representation.

In [15]:
# ============================================================
# 14. Identifier and entity discovery
# ============================================================

ENTITY_KEYWORDS = [
    "id",
    "user",
    "customer",
    "client",
    "card",
    "account",
    "merchant",
    "transaction"
]


def entity_candidates(df):
    """
    Identify possible entity/identifier columns.

    This version safely handles the case where no matching
    columns exist, which is expected for the ULB dataset
    because its anonymized variables are V1-V28.
    """

    rows = []

    for col in df.columns:
        low = str(col).lower()

        if any(k in low for k in ENTITY_KEYWORDS):

            rows.append({
                "column": col,
                "dtype": str(df[col].dtype),
                "unique": int(
                    df[col].nunique(dropna=True)
                ),
                "unique_ratio": float(
                    df[col].nunique(dropna=True) / len(df)
                )
            })

    # --------------------------------------------------------
    # IMPORTANT:
    # If no matching columns exist, return an empty DataFrame
    # with the expected schema instead of attempting to sort
    # by a nonexistent column.
    # --------------------------------------------------------

    if not rows:
        return pd.DataFrame(
            columns=[
                "column",
                "dtype",
                "unique",
                "unique_ratio"
            ]
        )

    return (
        pd.DataFrame(rows)
        .sort_values(
            "unique",
            ascending=False
        )
        .reset_index(drop=True)
    )


# ============================================================
# IBM entity candidates
# ============================================================

print("=" * 80)
print("IBM ENTITY CANDIDATES")
print("=" * 80)

ibm_entities = entity_candidates(ibm_sample)

if len(ibm_entities) > 0:
    display(ibm_entities)
else:
    print("No IBM entity/identifier candidates found.")


# ============================================================
# ULB entity candidates
# ============================================================

print("\n" + "=" * 80)
print("ULB ENTITY CANDIDATES")
print("=" * 80)

ulb_entities = entity_candidates(ulb_sample)

if len(ulb_entities) > 0:
    display(ulb_entities)
else:
    print(
        "No explicit ULB entity/identifier candidates found."
    )
    print(
        "This is expected because the ULB dataset uses "
        "anonymized/PCA-transformed features (V1-V28)."
    )

IBM ENTITY CANDIDATES


,column,dtype,unique,unique_ratio
0,Merchant Name,int64,407,0.0407
1,Merchant City,object,209,0.0209
2,Merchant State,object,37,0.0037
3,Card,int64,3,0.0003
4,User,int64,1,0.0001



ULB ENTITY CANDIDATES
No explicit ULB entity/identifier candidates found.
This is expected because the ULB dataset uses anonymized/PCA-transformed features (V1-V28).


## 15. IBM time/date field discovery

We need to identify whether IBM has:
- a transaction date
- a transaction timestamp
- an elapsed-time field
- month/year fields

Do not assume that a field called `Time` has the same meaning as ULB `Time`.

In [16]:
# ============================================================
# 15. Time/date candidates
# ============================================================

TIME_KEYWORDS = [
    "time",
    "date",
    "datetime",
    "timestamp",
    "year",
    "month",
    "day",
    "hour"
]

def time_candidates(df):
    rows = []

    for col in df.columns:
        low = col.lower()

        if any(k in low for k in TIME_KEYWORDS):
            rows.append({
                "column": col,
                "dtype": str(df[col].dtype),
                "sample_values": df[col].dropna().astype(str).head(5).tolist()
            })

    return pd.DataFrame(rows)


print("IBM time/date candidates:")
display(time_candidates(ibm_sample))

print("\nULB time/date fields:")
display(time_candidates(ulb_sample))

IBM time/date candidates:


,column,dtype,sample_values
0,Year,int64,"[2002, 2002, 2002, 2002, 2002]"
1,Month,int64,"[9, 9, 9, 9, 9]"
2,Day,int64,"[1, 1, 2, 2, 3]"
3,Time,object,"[06:21, 06:42, 06:22, 17:45, 06:23]"



ULB time/date fields:


,column,dtype,sample_values
0,Time,int64,"[0, 0, 1, 1, 2]"


In [17]:
# ============================================================
# 16. ULB time behavior
# ============================================================

print("ULB Time statistics:")
display(ulb_sample["Time"].describe())

print("\nULB Time range:")
print("min:", ulb_sample["Time"].min())
print("max:", ulb_sample["Time"].max())

print(
    "\nULB Time is an elapsed-time field measured in seconds "
    "from the beginning of the recorded period."
)

ULB Time statistics:


count    10000.000000
mean      5966.033400
std       4473.403739
min          0.000000
25%       2072.750000
50%       4563.500000
75%      10233.250000
max      15012.000000
Name: Time, dtype: float64


ULB Time range:
min: 0
max: 15012

ULB Time is an elapsed-time field measured in seconds from the beginning of the recorded period.


## 17. IBM cardinality analysis

High-cardinality fields can reveal the structure of the synthetic environment and help us identify whether user/card/merchant histories are available.

In [18]:
# ============================================================
# 17. IBM cardinality
# ============================================================

def cardinality_report(df):
    rows = []

    for col in df.columns:
        rows.append({
            "column": col,
            "dtype": str(df[col].dtype),
            "unique": int(df[col].nunique(dropna=True)),
            "unique_ratio": float(
                df[col].nunique(dropna=True) / len(df)
            )
        })

    return pd.DataFrame(rows).sort_values(
        "unique",
        ascending=False
    )


ibm_cardinality = cardinality_report(ibm_sample)
display(ibm_cardinality)

,column,dtype,unique,unique_ratio
6,Amount,object,6853,0.6853
5,Time,object,962,0.0962
8,Merchant Name,int64,407,0.0407
11,Zip,float64,283,0.0283
9,Merchant City,object,209,0.0209
12,MCC,int64,81,0.0081
10,Merchant State,object,37,0.0037
4,Day,int64,31,0.0031
2,Year,int64,19,0.0019
3,Month,int64,12,0.0012


## 18. Inspect the ULB fraud/legitimate amount distributions

`Amount` is one of the few ULB variables that retains direct financial meaning.

In [19]:
# ============================================================
# 18. ULB Amount by class
# ============================================================

for cls, label in [(0, "Legitimate"), (1, "Fraud")]:
    subset = ulb_sample.loc[
        ulb_sample[ULB_TARGET] == cls,
        "Amount"
    ]

    print(f"\nULB {label} transactions:")
    display(
        subset.describe(
            percentiles=[
                0.01, 0.05, 0.25, 0.50,
                0.75, 0.95, 0.99
            ]
        ).to_frame("Amount")
    )


ULB Legitimate transactions:


,Amount
count,9962.000000
mean,62.981743
std,183.901899
min,0.000000
1%,0.000000
5%,0.890000
25%,5.147500
50%,15.950000
75%,51.195000
95%,250.000000



ULB Fraud transactions:


,Amount
count,38.000000
mean,75.730526
std,304.521215
min,0.000000
1%,0.000000
5%,0.850000
25%,1.000000
50%,1.000000
75%,1.000000
95%,283.290500


In [20]:
# ============================================================
# 19. ULB numerical correlation with fraud
# ============================================================

ulb_numeric = ulb_sample.select_dtypes(
    include=np.number
)

if ULB_TARGET in ulb_numeric.columns:
    ulb_corr = (
        ulb_numeric
        .corr(numeric_only=True)[ULB_TARGET]
        .sort_values()
    )

    display(ulb_corr.to_frame("correlation_with_Class"))
else:
    print("ULB target is not numeric.")

,correlation_with_Class
V14,-0.517690
V17,-0.407361
V3,-0.390122
V12,-0.351070
V10,-0.344365
V16,-0.310866
V7,-0.212425
V9,-0.164610
V18,-0.129729
V6,-0.106040


## 20. IBM numerical correlation with fraud

This is only exploratory.

We will **not** infer causal relationships from correlation.

In [21]:
# ============================================================
# 20. IBM numerical correlation with fraud
# ============================================================

if IBM_TARGET is not None:

    ibm_numeric = ibm_sample.select_dtypes(
        include=np.number
    )

    if IBM_TARGET in ibm_numeric.columns:

        ibm_corr = (
            ibm_numeric
            .corr(numeric_only=True)[IBM_TARGET]
            .sort_values()
        )

        display(
            ibm_corr.to_frame(
                "correlation_with_IBM_target"
            )
        )

    else:
        print(
            f"{IBM_TARGET} is not a numerical column."
        )

else:
    print(
        "IBM_TARGET has not been identified yet."
    )

Is Fraud? is not a numerical column.


## 21. Duplicate analysis

We need to know whether duplicate transaction records exist in either dataset.

For the IBM 20M file, this is performed on a sample only at this stage.

In [22]:
# ============================================================
# 21. Duplicate analysis
# ============================================================

print("IBM sample duplicate rows:")
print(
    ibm_sample.duplicated().sum(),
    f"({ibm_sample.duplicated().mean()*100:.4f}%)"
)

print("\nULB duplicate rows:")
print(
    ulb_sample.duplicated().sum(),
    f"({ulb_sample.duplicated().mean()*100:.4f}%)"
)

IBM sample duplicate rows:
0 (0.0000%)

ULB duplicate rows:
42 (0.4200%)


## 22. Inspect likely identifier relationships among IBM auxiliary tables

If the card and user tables contain foreign keys that correspond to the transaction table, this cell will help identify possible joins.

We intentionally do not perform the joins automatically.

In [23]:
# ============================================================
# 22. Compare likely key names
# ============================================================

if ibm_cards_sample is not None:
    print("IBM transaction columns:")
    print(list(ibm_sample.columns))

    print("\nIBM cards columns:")
    print(list(ibm_cards_sample.columns))

if ibm_users_sample is not None:
    print("\nIBM users columns:")
    print(list(ibm_users_sample.columns))

IBM transaction columns:
['User', 'Card', 'Year', 'Month', 'Day', 'Time', 'Amount', 'Use Chip', 'Merchant Name', 'Merchant City', 'Merchant State', 'Zip', 'MCC', 'Errors?', 'Is Fraud?']

IBM cards columns:
['User', 'CARD INDEX', 'Card Brand', 'Card Type', 'Card Number', 'Expires', 'CVV', 'Has Chip', 'Cards Issued', 'Credit Limit', 'Acct Open Date', 'Year PIN last Changed', 'Card on Dark Web']

IBM users columns:
['Person', 'Current Age', 'Retirement Age', 'Birth Year', 'Birth Month', 'Gender', 'Address', 'Apartment', 'City', 'State', 'Zipcode', 'Latitude', 'Longitude', 'Per Capita Income - Zipcode', 'Yearly Income - Person', 'Total Debt', 'FICO Score', 'Num Credit Cards']


In [24]:
# ============================================================
# 23. Compare common column names
# ============================================================

def common_columns(df1, df2):
    a = {c.lower(): c for c in df1.columns}
    b = {c.lower(): c for c in df2.columns}

    common = sorted(set(a) & set(b))

    return pd.DataFrame({
        "normalized_name": common,
        "df1_column": [a[x] for x in common],
        "df2_column": [b[x] for x in common]
    })


if ibm_cards_sample is not None:
    print("Transaction ↔ Cards common columns:")
    display(
        common_columns(
            ibm_sample,
            ibm_cards_sample
        )
    )

if ibm_users_sample is not None:
    print("Transaction ↔ Users common columns:")
    display(
        common_columns(
            ibm_sample,
            ibm_users_sample
        )
    )

if ibm_cards_sample is not None and ibm_users_sample is not None:
    print("Cards ↔ Users common columns:")
    display(
        common_columns(
            ibm_cards_sample,
            ibm_users_sample
        )
    )

Transaction ↔ Cards common columns:


,normalized_name,df1_column,df2_column
0,user,User,User


Transaction ↔ Users common columns:


,normalized_name,df1_column,df2_column


Cards ↔ Users common columns:


,normalized_name,df1_column,df2_column


## 24. Large-file row count

This cell counts rows in the IBM main transaction CSV without loading the entire dataset into memory.

It can take some time for a multi-gigabyte file.

In [25]:
# ============================================================
# 24. Exact row counts
# ============================================================

def count_csv_rows(path):
    count = 0
    with open(path, "rb") as f:
        for _ in f:
            count += 1
    return max(count - 1, 0)


print("Counting IBM main transaction rows...")
IBM_ROWS = count_csv_rows(IBM_MAIN_PATH)

print(f"IBM main transaction rows: {IBM_ROWS:,}")

print("\nCounting ULB rows...")
ULB_ROWS = count_csv_rows(ULB_PATH)

print(f"ULB rows: {ULB_ROWS:,}")

Counting IBM main transaction rows...
IBM main transaction rows: 24,386,900

Counting ULB rows...
ULB rows: 284,807


## 25. Chunk-based IBM target distribution

If the IBM file contains a fraud label, we need its distribution over the **full dataset**, not only the 10,000-row sample.

This cell processes the file in chunks.

In [26]:
# ============================================================
# 25. Full IBM target distribution using chunks
# ============================================================

CHUNK_SIZE = 500_000

if IBM_TARGET is not None:

    target_counts = {}

    for chunk in pd.read_csv(
        IBM_MAIN_PATH,
        usecols=[IBM_TARGET],
        chunksize=CHUNK_SIZE,
        low_memory=False
    ):
        counts = chunk[IBM_TARGET].value_counts(dropna=False)

        for key, value in counts.items():
            target_counts[key] = (
                target_counts.get(key, 0) + int(value)
            )

    target_df = (
        pd.Series(target_counts, name="count")
        .to_frame()
    )

    target_df["percentage"] = (
        target_df["count"] /
        target_df["count"].sum() * 100
    )

    display(target_df)

else:
    print(
        "IBM_TARGET is not identified. "
        "Skipping full target scan."
    )

,count,percentage
No,24357143,99.87798
Yes,29757,0.12202


## 26. Full-file numeric ranges for the IBM main dataset

This uses chunks to calculate approximate global statistics without loading 20M+ rows into memory.

We first identify numerical columns from the sample.

In [27]:
# ============================================================
# 26. Chunk-based global numeric summary
# ============================================================

IBM_NUMERIC_COLUMNS = ibm_sample.select_dtypes(
    include=np.number
).columns.tolist()

print("IBM numerical columns:")
print(IBM_NUMERIC_COLUMNS)

# We calculate exact count/min/max and approximate moments from chunks.
global_stats = {}

for col in IBM_NUMERIC_COLUMNS:

    if col == IBM_TARGET:
        continue

    global_stats[col] = {
        "count": 0,
        "min": np.inf,
        "max": -np.inf,
        "sum": 0.0,
        "sum_sq": 0.0
    }


usecols = [
    c for c in IBM_NUMERIC_COLUMNS
    if c != IBM_TARGET
]

if usecols:
    for chunk in pd.read_csv(
        IBM_MAIN_PATH,
        usecols=usecols,
        chunksize=CHUNK_SIZE,
        low_memory=False
    ):
        for col in usecols:
            x = pd.to_numeric(
                chunk[col],
                errors="coerce"
            ).dropna()

            if len(x) == 0:
                continue

            arr = x.to_numpy(dtype=np.float64)

            global_stats[col]["count"] += len(arr)
            global_stats[col]["min"] = min(
                global_stats[col]["min"],
                float(arr.min())
            )
            global_stats[col]["max"] = max(
                global_stats[col]["max"],
                float(arr.max())
            )
            global_stats[col]["sum"] += float(arr.sum())
            global_stats[col]["sum_sq"] += float(
                np.square(arr).sum()
            )

    rows = []

    for col, s in global_stats.items():

        n = s["count"]

        if n > 0:
            mean = s["sum"] / n
            variance = max(
                s["sum_sq"] / n - mean**2,
                0
            )
            std = np.sqrt(variance)
        else:
            mean = np.nan
            std = np.nan

        rows.append({
            "column": col,
            "count": n,
            "mean": mean,
            "std": std,
            "min": s["min"],
            "max": s["max"]
        })

    global_numeric_summary = pd.DataFrame(rows)

    display(global_numeric_summary)

else:
    print("No IBM numerical columns available.")

IBM numerical columns:
['User', 'Card', 'Year', 'Month', 'Day', 'Merchant Name', 'Zip', 'MCC']


,column,count,mean,std,min,max
0,User,24386900,1.001019e+03,5.694611e+02,0.000000e+00,1.999000e+03
1,Card,24386900,1.351366e+00,1.407154e+00,0.000000e+00,8.000000e+00
2,Year,24386900,2.011955e+03,5.105921e+00,1.991000e+03,2.020000e+03
3,Month,24386900,6.525064e+00,3.472355e+00,1.000000e+00,1.200000e+01
4,Day,24386900,1.571812e+01,8.794073e+00,1.000000e+00,3.100000e+01
5,Merchant Name,24386900,-4.769230e+17,4.758940e+18,-9.222899e+18,9.223292e+18
6,Zip,21508765,5.095644e+04,2.939707e+04,5.010000e+02,9.992800e+04
7,MCC,24386900,5.561171e+03,8.793154e+02,1.711000e+03,9.402000e+03


## 27. Check whether IBM has repeated users/cards/merchants

If an entity identifier exists, this chunked scan can calculate the number of unique entities and transaction frequency statistics.

The notebook tries likely identifier columns automatically.

In [28]:
# ============================================================
# 27. Candidate repeated entities in IBM
# ============================================================

possible_entity_cols = []

for col in ibm_sample.columns:
    low = col.lower()

    if any(
        k in low
        for k in [
            "user",
            "customer",
            "client",
            "card",
            "account",
            "merchant"
        ]
    ):
        possible_entity_cols.append(col)

print("Possible entity columns:")
print(possible_entity_cols)

# Exact unique counts can require a large memory footprint.
# Therefore we estimate frequency structure from the first sample.
for col in possible_entity_cols:

    counts = ibm_sample[col].value_counts(
        dropna=False
    )

    print(f"\n--- {col} ---")
    print("Sample unique entities:", len(counts))
    print(
        "Sample median transactions/entity:",
        counts.median()
    )
    print(
        "Sample mean transactions/entity:",
        counts.mean()
    )
    print(
        "Sample maximum transactions/entity:",
        counts.max()
    )

Possible entity columns:
['User', 'Card', 'Merchant Name', 'Merchant City', 'Merchant State']

--- User ---
Sample unique entities: 1
Sample median transactions/entity: 10000.0
Sample mean transactions/entity: 10000.0
Sample maximum transactions/entity: 10000

--- Card ---
Sample unique entities: 3
Sample median transactions/entity: 3786.0
Sample mean transactions/entity: 3333.3333333333335
Sample maximum transactions/entity: 5011

--- Merchant Name ---
Sample unique entities: 407
Sample median transactions/entity: 1.0
Sample mean transactions/entity: 24.57002457002457
Sample maximum transactions/entity: 941

--- Merchant City ---
Sample unique entities: 209
Sample median transactions/entity: 3.0
Sample mean transactions/entity: 47.84688995215311
Sample maximum transactions/entity: 5440

--- Merchant State ---
Sample unique entities: 38
Sample median transactions/entity: 15.5
Sample mean transactions/entity: 263.1578947368421
Sample maximum transactions/entity: 8603


## 28. Build a preliminary semantic-feature checklist

This does **not** create features yet.

It simply checks whether the raw datasets appear to contain information from which common behavioral concepts might be derived.

Potential shared concepts:

- transaction amount
- transaction time
- transaction frequency / velocity
- amount deviation from historical behavior
- temporal irregularity
- merchant novelty
- card/customer activity
- transaction count in a recent window

The final mapping must be decided after reviewing the actual IBM schema.

In [29]:
# ============================================================
# 28. Preliminary semantic feature availability
# ============================================================

def find_matching_columns(columns, keywords):
    matches = []

    for col in columns:
        low = col.lower()

        if any(k in low for k in keywords):
            matches.append(col)

    return matches


semantic_groups = {
    "amount": ["amount", "price", "value", "cost"],
    "time": ["time", "date", "timestamp", "datetime", "hour"],
    "user": ["user", "customer", "client", "person"],
    "card": ["card", "account"],
    "merchant": ["merchant", "store", "vendor"],
    "location": ["location", "city", "state", "country", "zip", "lat", "lon"],
    "category": ["category", "mcc", "merchant_category", "type"],
    "transaction_id": ["transaction_id", "trans_id", "txn_id"]
}

rows = []

for group, keywords in semantic_groups.items():

    rows.append({
        "semantic_group": group,
        "IBM_matches": find_matching_columns(
            ibm_sample.columns,
            keywords
        ),
        "ULB_matches": find_matching_columns(
            ulb_sample.columns,
            keywords
        )
    })

semantic_report = pd.DataFrame(rows)

display(semantic_report)

,semantic_group,IBM_matches,ULB_matches
0,amount,[Amount],[Amount]
1,time,[Time],[Time]
2,user,[User],[]
3,card,[Card],[]
4,merchant,"[Merchant Name, Merchant City, Merchant State]",[]
5,location,"[Merchant City, Merchant State, Zip]",[]
6,category,[MCC],[]
7,transaction_id,[],[]


## 29. ULB PCA-feature reminder

The ULB variables `V1`–`V28` are anonymized/PCA-transformed features.

For the eventual paper:

**Do not assign semantic names to individual PCA components.**

We can use them statistically, but if we propose a cross-dataset semantic alignment method, we should only map concepts that have legitimate support in both datasets.

In [30]:
# ============================================================
# 29. Confirm ULB feature structure
# ============================================================

ulb_pca_cols = [
    c for c in ulb_sample.columns
    if re.fullmatch(r"V\d+", str(c))
]

print("ULB PCA-style columns:")
print(ulb_pca_cols)

print("\nCount:", len(ulb_pca_cols))

print("\nOther ULB columns:")
print([
    c for c in ulb_sample.columns
    if c not in ulb_pca_cols
])

ULB PCA-style columns:
['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20', 'V21', 'V22', 'V23', 'V24', 'V25', 'V26', 'V27', 'V28']

Count: 28

Other ULB columns:
['Time', 'Amount', 'Class']


## 30. Save the complete reconnaissance report

The JSON file stores schema-level information so the notebook can be reproduced without sending the datasets themselves.

In [31]:
# ============================================================
# 30. Save reconnaissance metadata
# ============================================================

recon = {
    "paths": {
        "ULB": ULB_PATH,
        "IBM_MAIN": IBM_MAIN_PATH,
        "IBM_USER0": IBM_USER0_PATH,
        "IBM_CARDS": IBM_CARDS_PATH,
        "IBM_USERS": IBM_USERS_PATH
    },

    "row_counts": {
        "IBM_MAIN": int(IBM_ROWS),
        "ULB": int(ULB_ROWS)
    },

    "IBM_columns": list(ibm_sample.columns),
    "ULB_columns": list(ulb_sample.columns),

    "IBM_dtypes": {
        c: str(ibm_sample[c].dtype)
        for c in ibm_sample.columns
    },

    "ULB_dtypes": {
        c: str(ulb_sample[c].dtype)
        for c in ulb_sample.columns
    },

    "IBM_target_candidates": target_candidates(ibm_sample),
    "ULB_target_candidates": target_candidates(ulb_sample),

    "IBM_entity_candidates": (
        ibm_entities.to_dict(orient="records")
        if len(ibm_entities)
        else []
    ),

    "ULB_entity_candidates": (
        ulb_entities.to_dict(orient="records")
        if len(ulb_entities)
        else []
    ),

    "IBM_time_candidates": (
        time_candidates(ibm_sample).to_dict(
            orient="records"
        )
    ),

    "ULB_time_candidates": (
        time_candidates(ulb_sample).to_dict(
            orient="records"
        )
    )
}

with open(
    "credit_card_dataset_reconnaissance.json",
    "w"
) as f:
    json.dump(
        recon,
        f,
        indent=2,
        default=str
    )

print(
    "Saved: credit_card_dataset_reconnaissance.json"
)

Saved: credit_card_dataset_reconnaissance.json


# 31. What to send back

After running the notebook, send the outputs of these sections:

### Most important
1. **IBM MAIN TRANSACTION FILE — first rows and columns**
2. **IBM auxiliary schemas**
3. **IBM target candidates/distribution**
4. **IBM entity candidates**
5. **IBM time/date candidates**
6. **IBM cardinality report**
7. **IBM numerical summary**
8. **ULB columns and first rows**
9. **ULB target distribution**
10. **Semantic feature availability table**

### Especially useful

If the notebook produces:

`credit_card_dataset_reconnaissance.json`

you can also upload that small JSON file.

### Do not train anything yet.

The next stage will be decided from the actual schema. We need to verify that the proposed synthetic-to-real transfer representation is scientifically defensible before implementing the model.

In [32]:
# ============================================================
# FULL IBM ENTITY + LABEL RECONNAISSANCE
# ============================================================

import pandas as pd
import numpy as np
from collections import Counter

CHUNK_SIZE = 500_000

# Columns we specifically need for the full scan
ENTITY_COLS = [
    "User",
    "Card",
    "Merchant Name",
    "Merchant City",
    "Merchant State",
    "MCC",
    "Is Fraud?"
]

global_unique = {
    col: set()
    for col in ENTITY_COLS
}

fraud_counter = Counter()

total_rows = 0

print("Starting full IBM scan...")
print(f"Chunk size: {CHUNK_SIZE:,}")
print("=" * 80)

for chunk_no, chunk in enumerate(
    pd.read_csv(
        IBM_MAIN_PATH,
        usecols=ENTITY_COLS,
        chunksize=CHUNK_SIZE,
        low_memory=False
    ),
    start=1
):

    total_rows += len(chunk)

    # --------------------------------------------------------
    # Unique entity values
    # --------------------------------------------------------

    for col in ENTITY_COLS:

        if col == "Is Fraud?":
            continue

        values = chunk[col].dropna().unique()

        global_unique[col].update(values.tolist())

    # --------------------------------------------------------
    # Fraud labels
    # --------------------------------------------------------

    fraud_counter.update(
        chunk["Is Fraud?"]
        .astype(str)
        .str.strip()
        .str.lower()
        .value_counts()
        .to_dict()
    )

    if chunk_no % 5 == 0:
        print(
            f"Processed: {total_rows:,} rows | "
            f"Chunks: {chunk_no}"
        )

print("\n" + "=" * 80)
print("FULL IBM SCAN COMPLETE")
print("=" * 80)

print(f"Total rows processed: {total_rows:,}")

# ============================================================
# Entity cardinalities
# ============================================================

entity_results = []

for col, values in global_unique.items():

    entity_results.append({
        "column": col,
        "full_dataset_unique": len(values)
    })

entity_results = pd.DataFrame(
    entity_results
).sort_values(
    "full_dataset_unique",
    ascending=False
).reset_index(drop=True)

print("\nFULL DATASET ENTITY CARDINALITY")
display(entity_results)

# ============================================================
# Fraud distribution
# ============================================================

fraud_results = (
    pd.Series(
        fraud_counter,
        name="count"
    )
    .to_frame()
)

fraud_results["percentage"] = (
    fraud_results["count"]
    / fraud_results["count"].sum()
    * 100
)

print("\nFULL IBM FRAUD LABEL DISTRIBUTION")
display(fraud_results)

Starting full IBM scan...
Chunk size: 500,000
Processed: 2,500,000 rows | Chunks: 5
Processed: 5,000,000 rows | Chunks: 10
Processed: 7,500,000 rows | Chunks: 15
Processed: 10,000,000 rows | Chunks: 20
Processed: 12,500,000 rows | Chunks: 25
Processed: 15,000,000 rows | Chunks: 30
Processed: 17,500,000 rows | Chunks: 35
Processed: 20,000,000 rows | Chunks: 40
Processed: 22,500,000 rows | Chunks: 45

FULL IBM SCAN COMPLETE
Total rows processed: 24,386,900

FULL DATASET ENTITY CARDINALITY


,column,full_dataset_unique
0,Merchant Name,100343
1,Merchant City,13429
2,User,2000
3,Merchant State,223
4,MCC,109
5,Card,9
6,Is Fraud?,0



FULL IBM FRAUD LABEL DISTRIBUTION


,count,percentage
no,24357143,99.87798
yes,29757,0.12202


In [33]:
# ============================================================
# IBM USER / CARD TRANSACTION FREQUENCY
# ============================================================

FREQ_COLS = [
    "User",
    "Card",
    "Merchant Name"
]

frequency_counters = {
    col: Counter()
    for col in FREQ_COLS
}

total_rows = 0

print("Calculating full-dataset frequency distributions...")

for chunk in pd.read_csv(
    IBM_MAIN_PATH,
    usecols=FREQ_COLS,
    chunksize=CHUNK_SIZE,
    low_memory=False
):

    total_rows += len(chunk)

    for col in FREQ_COLS:

        counts = chunk[col].value_counts(
            dropna=True
        )

        frequency_counters[col].update(
            counts.to_dict()
        )

print("\n" + "=" * 80)
print("FREQUENCY ANALYSIS")
print("=" * 80)

for col, counter in frequency_counters.items():

    values = np.array(
        list(counter.values()),
        dtype=np.int64
    )

    print(f"\n--- {col} ---")

    print(
        "Number of unique entities:",
        len(values)
    )

    print(
        "Min transactions/entity:",
        values.min()
    )

    print(
        "Median transactions/entity:",
        np.median(values)
    )

    print(
        "Mean transactions/entity:",
        values.mean()
    )

    print(
        "95th percentile:",
        np.percentile(values, 95)
    )

    print(
        "Maximum transactions/entity:",
        values.max()
    )

Calculating full-dataset frequency distributions...

FREQUENCY ANALYSIS

--- User ---
Number of unique entities: 2000
Min transactions/entity: 15
Median transactions/entity: 10860.5
Mean transactions/entity: 12193.45
95th percentile: 31473.09999999999
Maximum transactions/entity: 82355

--- Card ---
Number of unique entities: 9
Min transactions/entity: 5184
Median transactions/entity: 1309120.0
Mean transactions/entity: 2709655.5555555555
95th percentile: 7815285.399999999
Maximum transactions/entity: 8696411

--- Merchant Name ---
Number of unique entities: 100343
Min transactions/entity: 1
Median transactions/entity: 4.0
Mean transactions/entity: 243.03538861704354
95th percentile: 319.0
Maximum transactions/entity: 1130230


In [34]:
# ============================================================
# IBM FRAUD RATE BY USE-CHIP / MCC / CARD
# ============================================================

GROUP_COLS = [
    "Card",
    "MCC",
    "Use Chip"
]

fraud_stats = {}

for col in GROUP_COLS:

    print("\n" + "=" * 80)
    print(f"FRAUD RATE BY: {col}")
    print("=" * 80)

    stats = []

    for chunk in pd.read_csv(
        IBM_MAIN_PATH,
        usecols=[
            col,
            "Is Fraud?"
        ],
        chunksize=CHUNK_SIZE,
        low_memory=False
    ):

        temp = chunk.copy()

        # Normalize fraud label
        temp["_fraud"] = (
            temp["Is Fraud?"]
            .astype(str)
            .str.strip()
            .str.lower()
            .isin([
                "yes",
                "true",
                "1",
                "fraud",
                "y"
            ])
        )

        grouped = (
            temp
            .groupby(col)["_fraud"]
            .agg(
                transactions="count",
                fraud_count="sum"
            )
        )

        stats.append(grouped)

    stats = pd.concat(stats)

    stats = (
        stats
        .groupby(level=0)
        .sum()
    )

    stats["fraud_rate"] = (
        stats["fraud_count"]
        / stats["transactions"]
        * 100
    )

    stats = stats.sort_values(
        "fraud_rate",
        ascending=False
    )

    display(stats.head(30))

    fraud_stats[col] = stats


FRAUD RATE BY: Card


,transactions,fraud_count,fraud_rate
Card,,,
8,5184,25,0.482253
7,46383,109,0.235000
6,176729,350,0.198043
5,563097,1016,0.180431
4,1309120,2157,0.164767
3,2790785,4135,0.148166
2,4305594,5807,0.134871
1,6493597,7514,0.115714
0,8696411,8644,0.099397



FRAUD RATE BY: MCC


,transactions,fraud_count,fraud_rate
MCC,,,
4411,634,317,50.000000
5733,496,127,25.604839
5045,5013,470,9.375623
3006,702,59,8.404558
3144,632,53,8.386076
3008,714,50,7.002801
5732,12593,843,6.694195
3007,666,41,6.156156
3075,666,40,6.006006



FRAUD RATE BY: Use Chip


,transactions,fraud_count,fraud_rate
Use Chip,,,
Online Transaction,2713220,18349,0.676281
Chip Transaction,6287598,4836,0.076913
Swipe Transaction,15386082,6572,0.042714


In [37]:
# ============================================================
# 32-FIX. ROBUST BEHAVIORAL FEATURE CONSTRUCTION
# ============================================================

import numpy as np
import pandas as pd

print("=" * 80)
print("REBUILDING BEHAVIORAL FEATURES")
print("=" * 80)

# ------------------------------------------------------------
# Check what we already have
# ------------------------------------------------------------

print("Current columns:")
print(behavior_df.columns.tolist())

# ------------------------------------------------------------
# Ensure correct ordering
# ------------------------------------------------------------

behavior_df = behavior_df.sort_values(
    ["User", "Timestamp"],
    kind="mergesort"
).reset_index(drop=True)

# ------------------------------------------------------------
# Make sure Amount is numeric
# ------------------------------------------------------------

behavior_df["Amount"] = pd.to_numeric(
    behavior_df["Amount"],
    errors="coerce"
)

# ------------------------------------------------------------
# 1. Inter-transaction time
# ------------------------------------------------------------

if "Delta_t_seconds" not in behavior_df.columns:

    behavior_df["Delta_t_seconds"] = (
        behavior_df
        .groupby("User")["Timestamp"]
        .diff()
        .dt.total_seconds()
    )

behavior_df.loc[
    behavior_df["Delta_t_seconds"] < 0,
    "Delta_t_seconds"
] = np.nan

# ------------------------------------------------------------
# 2. Previous transaction amount
# ------------------------------------------------------------

behavior_df["Previous_amount"] = (
    behavior_df
    .groupby("User")["Amount"]
    .shift(1)
)

# ------------------------------------------------------------
# 3. Amount change from previous transaction
# ------------------------------------------------------------

behavior_df["Amount_change"] = (
    behavior_df["Amount"]
    - behavior_df["Previous_amount"]
)

behavior_df["Amount_change_abs"] = (
    behavior_df["Amount_change"].abs()
)

# ------------------------------------------------------------
# 4. Rolling amount mean
#
# Uses ONLY previous transactions.
# This prevents target transaction leakage.
# ------------------------------------------------------------

behavior_df["Rolling_amount_mean"] = (
    behavior_df
    .groupby("User")["Amount"]
    .transform(
        lambda x:
        x.shift(1)
        .rolling(
            window=20,
            min_periods=3
        )
        .mean()
    )
)

# ------------------------------------------------------------
# 5. Rolling amount standard deviation
# ------------------------------------------------------------

behavior_df["Rolling_amount_std"] = (
    behavior_df
    .groupby("User")["Amount"]
    .transform(
        lambda x:
        x.shift(1)
        .rolling(
            window=20,
            min_periods=3
        )
        .std()
    )
)

# ------------------------------------------------------------
# 6. Standardized amount deviation
# ------------------------------------------------------------

behavior_df["Amount_deviation"] = (
    (
        behavior_df["Amount"]
        - behavior_df["Rolling_amount_mean"]
    )
    /
    (
        behavior_df["Rolling_amount_std"]
        + 1e-6
    )
).abs()

# ------------------------------------------------------------
# 7. Relative amount change
# ------------------------------------------------------------

behavior_df["Amount_ratio"] = (
    behavior_df["Amount"]
    /
    (
        behavior_df["Rolling_amount_mean"].abs()
        + 1e-6
    )
)

# ------------------------------------------------------------
# 8. Recent transaction count
#
# Number of previous transactions among the last 20
# transactions for the same user.
# ------------------------------------------------------------

behavior_df["Recent_transaction_count"] = (
    behavior_df
    .groupby("User")["Amount"]
    .transform(
        lambda x:
        x.shift(1)
        .rolling(
            window=20,
            min_periods=1
        )
        .count()
    )
)

# ------------------------------------------------------------
# 9. Temporal velocity
#
# Reciprocal of inter-transaction time.
# Small Delta_t -> high velocity.
# ------------------------------------------------------------

behavior_df["Transaction_velocity"] = (
    1.0
    /
    (
        behavior_df["Delta_t_seconds"]
        + 1.0
    )
)

# ------------------------------------------------------------
# 10. Hour and day-of-week
# ------------------------------------------------------------

behavior_df["Hour"] = (
    behavior_df["Timestamp"].dt.hour
)

behavior_df["DayOfWeek"] = (
    behavior_df["Timestamp"].dt.dayofweek
)

# ------------------------------------------------------------
# 11. Verification
# ------------------------------------------------------------

NEW_BEHAVIOR_COLS = [
    "Amount",
    "Delta_t_seconds",
    "Previous_amount",
    "Amount_change",
    "Amount_change_abs",
    "Rolling_amount_mean",
    "Rolling_amount_std",
    "Amount_deviation",
    "Amount_ratio",
    "Recent_transaction_count",
    "Transaction_velocity",
    "Hour",
    "DayOfWeek"
]

print("\n" + "=" * 80)
print("FEATURE VERIFICATION")
print("=" * 80)

verification = pd.DataFrame({
    "feature": NEW_BEHAVIOR_COLS,
    "exists": [
        col in behavior_df.columns
        for col in NEW_BEHAVIOR_COLS
    ],
    "non_null": [
        int(behavior_df[col].notna().sum())
        if col in behavior_df.columns else 0
        for col in NEW_BEHAVIOR_COLS
    ]
})

verification["non_null_pct"] = (
    verification["non_null"]
    / len(behavior_df)
    * 100
)

display(verification)

print("\nBehavioral dataframe shape:")
print(behavior_df.shape)

REBUILDING BEHAVIORAL FEATURES
Current columns:
['User', 'Year', 'Month', 'Day', 'Time', 'Amount', 'Is Fraud?', 'Fraud', 'Timestamp', 'Delta_t_seconds']

FEATURE VERIFICATION


,feature,exists,non_null,non_null_pct
0,Amount,True,0,0.000000
1,Delta_t_seconds,True,24384900,99.991799
2,Previous_amount,True,0,0.000000
3,Amount_change,True,0,0.000000
4,Amount_change_abs,True,0,0.000000
5,Rolling_amount_mean,True,0,0.000000
6,Rolling_amount_std,True,0,0.000000
7,Amount_deviation,True,0,0.000000
8,Amount_ratio,True,0,0.000000
9,Recent_transaction_count,True,24386900,100.000000



Behavioral dataframe shape:
(24386900, 21)


In [38]:
# ============================================================
# 33. FRAUD RATE ACROSS BEHAVIORAL BINS
# ============================================================

def fraud_rate_by_quantile(
    df,
    feature,
    n_bins=10
):

    temp = df[
        [feature, "Fraud"]
    ].dropna().copy()

    try:
        temp["bin"] = pd.qcut(
            temp[feature],
            q=n_bins,
            duplicates="drop"
        )
    except Exception:
        return pd.DataFrame()

    result = (
        temp
        .groupby("bin", observed=True)["Fraud"]
        .agg(
            transactions="count",
            fraud_count="sum",
            fraud_rate="mean"
        )
    )

    result["fraud_rate"] *= 100

    return result


for feature in [
    "Amount",
    "Delta_t_seconds",
    "Amount_deviation"
]:

    print("\n" + "=" * 80)
    print(f"FRAUD RATE BY {feature}")
    print("=" * 80)

    display(
        fraud_rate_by_quantile(
            behavior_df,
            feature
        )
    )


FRAUD RATE BY Amount


,transactions,fraud_count,fraud_rate
bin,,,



FRAUD RATE BY Delta_t_seconds


,transactions,fraud_count,fraud_rate
bin,,,
"(-0.001, 600.0]",2577789,4420,0.171465
"(600.0, 1800.0]",2345575,4777,0.203660
"(1800.0, 4080.0]",2438220,4163,0.170739
"(4080.0, 8280.0]",2418563,4775,0.197431
"(8280.0, 13920.0]",2422663,3375,0.139310
"(13920.0, 21960.0]",2428528,2226,0.091660
"(21960.0, 35040.0]",2445184,1382,0.056519
"(35040.0, 53340.0]",2434432,1491,0.061246
"(53340.0, 76680.0]",2435886,1869,0.076728



FRAUD RATE BY Amount_deviation


,transactions,fraud_count,fraud_rate
bin,,,


In [39]:
# ============================================================
# 34. TEMPORAL FRAUD PATTERNS
# ============================================================

print("=" * 80)
print("FRAUD RATE BY HOUR")
print("=" * 80)

hour_stats = (
    behavior_df
    .groupby("Hour")["Fraud"]
    .agg(
        transactions="count",
        fraud_count="sum",
        fraud_rate="mean"
    )
)

hour_stats["fraud_rate"] *= 100

display(hour_stats)


print("\n" + "=" * 80)
print("FRAUD RATE BY DAY OF WEEK")
print("=" * 80)

dow_stats = (
    behavior_df
    .groupby("DayOfWeek")["Fraud"]
    .agg(
        transactions="count",
        fraud_count="sum",
        fraud_rate="mean"
    )
)

dow_stats["fraud_rate"] *= 100

display(dow_stats)

FRAUD RATE BY HOUR


,transactions,fraud_count,fraud_rate
Hour,,,
0,235608,120,0.050932
1,213411,185,0.086687
2,214075,231,0.107906
3,183370,342,0.186508
4,203550,372,0.182756
5,321588,505,0.157033
6,1347058,855,0.063472
7,1644618,1597,0.097105
8,1616851,1674,0.103535



FRAUD RATE BY DAY OF WEEK


,transactions,fraud_count,fraud_rate
DayOfWeek,,,
0,3480263,3876,0.111371
1,3468169,3921,0.113057
2,3475771,2911,0.083751
3,3495004,5220,0.149356
4,3482897,4908,0.140917
5,3499591,3506,0.100183
6,3485205,5415,0.155371


In [40]:
# ============================================================
# 35. DIAGNOSE IBM AMOUNT COLUMN
# ============================================================

print("=" * 80)
print("IBM AMOUNT DIAGNOSTICS")
print("=" * 80)

# Basic information
print("\nColumn dtype:")
print(behavior_df["Amount"].dtype)

print("\nFirst 30 raw values:")
display(
    behavior_df["Amount"].head(30)
)

print("\nRaw unique examples:")
display(
    behavior_df["Amount"]
    .drop_duplicates()
    .head(30)
)

print("\nMissing values:")
print(
    behavior_df["Amount"].isna().sum()
)

print("\nNon-missing values:")
print(
    behavior_df["Amount"].notna().sum()
)

print("\nNumber of unique values:")
print(
    behavior_df["Amount"].nunique(
        dropna=True
    )
)

print("\nValue type distribution:")

display(
    behavior_df["Amount"]
    .map(type)
    .value_counts()
)

# ------------------------------------------------------------
# Try string cleaning
# ------------------------------------------------------------

amount_test = (
    behavior_df["Amount"]
    .astype(str)
    .str.strip()
)

print("\nExamples after string conversion:")
display(
    amount_test.head(30)
)

# ------------------------------------------------------------
# Remove common currency formatting
# ------------------------------------------------------------

amount_clean = (
    amount_test
    .str.replace("$", "", regex=False)
    .str.replace(",", "", regex=False)
    .str.strip()
)

amount_numeric = pd.to_numeric(
    amount_clean,
    errors="coerce"
)

print("\nSuccessfully converted to numeric:")
print(
    amount_numeric.notna().sum()
)

print("\nConversion failure:")
print(
    amount_numeric.isna().sum()
)

print("\nConversion success rate:")
print(
    amount_numeric.notna().mean() * 100
)

print("\nNumeric statistics:")
display(
    amount_numeric.describe(
        percentiles=[
            0.01,
            0.05,
            0.25,
            0.50,
            0.75,
            0.95,
            0.99
        ]
    )
)

IBM AMOUNT DIAGNOSTICS

Column dtype:
float64

First 30 raw values:


0    NaN
1    NaN
2    NaN
3    NaN
4    NaN
5    NaN
6    NaN
7    NaN
8    NaN
9    NaN
10   NaN
11   NaN
12   NaN
13   NaN
14   NaN
15   NaN
16   NaN
17   NaN
18   NaN
19   NaN
20   NaN
21   NaN
22   NaN
23   NaN
24   NaN
25   NaN
26   NaN
27   NaN
28   NaN
29   NaN
Name: Amount, dtype: float64


Raw unique examples:


0   NaN
Name: Amount, dtype: float64


Missing values:
24386900

Non-missing values:
0

Number of unique values:
0

Value type distribution:


Amount
<class 'float'>    24386900
Name: count, dtype: int64


Examples after string conversion:


0     nan
1     nan
2     nan
3     nan
4     nan
5     nan
6     nan
7     nan
8     nan
9     nan
10    nan
11    nan
12    nan
13    nan
14    nan
15    nan
16    nan
17    nan
18    nan
19    nan
20    nan
21    nan
22    nan
23    nan
24    nan
25    nan
26    nan
27    nan
28    nan
29    nan
Name: Amount, dtype: object


Successfully converted to numeric:
0

Conversion failure:
24386900

Conversion success rate:
0.0

Numeric statistics:


count    0.0
mean     NaN
std      NaN
min      NaN
1%       NaN
5%       NaN
25%      NaN
50%      NaN
75%      NaN
95%      NaN
99%      NaN
max      NaN
Name: Amount, dtype: float64

In [41]:
# ============================================================
# 36. DIRECT IBM AMOUNT VALIDATION
# ============================================================

import pandas as pd
import numpy as np

print("=" * 80)
print("DIRECT VALIDATION OF IBM AMOUNT")
print("=" * 80)

# ------------------------------------------------------------
# Read a small sample DIRECTLY from the original CSV
# ------------------------------------------------------------

amount_check = pd.read_csv(
    IBM_MAIN_PATH,
    usecols=[
        "User",
        "Year",
        "Month",
        "Day",
        "Time",
        "Amount",
        "Is Fraud?"
    ],
    nrows=10000,
    low_memory=False
)

print("\nShape:")
print(amount_check.shape)

print("\nDtypes:")
print(amount_check.dtypes)

print("\nFirst 20 rows:")
display(amount_check.head(20))

print("\nAmount missing:")
print(
    amount_check["Amount"].isna().sum()
)

print("\nAmount non-missing:")
print(
    amount_check["Amount"].notna().sum()
)

print("\nAmount unique:")
print(
    amount_check["Amount"].nunique(
        dropna=True
    )
)

print("\nAmount examples:")
display(
    amount_check["Amount"]
    .dropna()
    .head(30)
)

print("\nFull descriptive statistics:")
display(
    amount_check["Amount"].describe()
)

DIRECT VALIDATION OF IBM AMOUNT

Shape:
(10000, 7)

Dtypes:
User          int64
Year          int64
Month         int64
Day           int64
Time         object
Amount       object
Is Fraud?    object
dtype: object

First 20 rows:


,User,Year,Month,Day,Time,Amount,Is Fraud?
0,0,2002,9,1,06:21,$134.09,No
1,0,2002,9,1,06:42,$38.48,No
2,0,2002,9,2,06:22,$120.34,No
3,0,2002,9,2,17:45,$128.95,No
4,0,2002,9,3,06:23,$104.71,No
5,0,2002,9,3,13:53,$86.19,No
6,0,2002,9,4,05:51,$93.84,No
7,0,2002,9,4,06:09,$123.50,No
8,0,2002,9,5,06:14,$61.72,No
9,0,2002,9,5,09:35,$57.10,No



Amount missing:
0

Amount non-missing:
10000

Amount unique:
6853

Amount examples:


0     $134.09
1      $38.48
2     $120.34
3     $128.95
4     $104.71
5      $86.19
6      $93.84
7     $123.50
8      $61.72
9      $57.10
10     $76.07
11     $53.91
12    $110.37
13    $117.05
14     $45.30
15     $29.34
16    $147.45
17     $27.75
18     $76.57
19     $22.56
20     $37.50
21     $65.50
22     $56.42
23      $2.71
24    $144.90
25    $160.00
26    $102.18
27     $36.73
28     $29.33
29    $162.39
Name: Amount, dtype: object


Full descriptive statistics:


count       10000
unique       6853
top       $140.00
freq           34
Name: Amount, dtype: object

In [42]:
# ============================================================
# 37. IBM AMOUNT MISSINGNESS ACROSS THE FULL DATASET
# ============================================================

CHUNK_SIZE = 500_000

total_rows = 0
amount_nonmissing = 0
amount_missing = 0

amount_min = np.inf
amount_max = -np.inf

print("=" * 80)
print("FULL IBM AMOUNT VALIDATION")
print("=" * 80)

for chunk_no, chunk in enumerate(
    pd.read_csv(
        IBM_MAIN_PATH,
        usecols=["Amount"],
        chunksize=CHUNK_SIZE,
        low_memory=False
    ),
    start=1
):

    total_rows += len(chunk)

    valid = pd.to_numeric(
        chunk["Amount"],
        errors="coerce"
    )

    amount_nonmissing += valid.notna().sum()
    amount_missing += valid.isna().sum()

    if valid.notna().any():
        amount_min = min(
            amount_min,
            valid.min()
        )

        amount_max = max(
            amount_max,
            valid.max()
        )

    if chunk_no % 5 == 0:
        print(
            f"Processed: {total_rows:,} rows"
        )

print("\n" + "=" * 80)
print("RESULT")
print("=" * 80)

print(f"Total rows:       {total_rows:,}")
print(f"Non-missing:      {amount_nonmissing:,}")
print(f"Missing:          {amount_missing:,}")
print(
    f"Missing %:        "
    f"{amount_missing / total_rows * 100:.4f}%"
)

if amount_nonmissing > 0:
    print(f"Minimum amount:   {amount_min}")
    print(f"Maximum amount:   {amount_max}")
else:
    print("No numeric Amount values found.")

FULL IBM AMOUNT VALIDATION
Processed: 2,500,000 rows
Processed: 5,000,000 rows
Processed: 7,500,000 rows
Processed: 10,000,000 rows
Processed: 12,500,000 rows
Processed: 15,000,000 rows
Processed: 17,500,000 rows
Processed: 20,000,000 rows
Processed: 22,500,000 rows

RESULT
Total rows:       24,386,900
Non-missing:      0
Missing:          24,386,900
Missing %:        100.0000%
No numeric Amount values found.


In [43]:
# ============================================================
# 37-FIX. FULL IBM AMOUNT VALIDATION
# ============================================================

CHUNK_SIZE = 500_000

total_rows = 0
amount_nonmissing = 0
amount_missing = 0

amount_min = np.inf
amount_max = -np.inf

print("=" * 80)
print("FULL IBM AMOUNT VALIDATION - FIXED")
print("=" * 80)

for chunk_no, chunk in enumerate(
    pd.read_csv(
        IBM_MAIN_PATH,
        usecols=["Amount"],
        chunksize=CHUNK_SIZE,
        low_memory=False
    ),
    start=1
):

    total_rows += len(chunk)

    # IBM amounts are strings such as "$134.09"
    valid = pd.to_numeric(
        chunk["Amount"]
        .astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
        .str.strip(),
        errors="coerce"
    )

    amount_nonmissing += int(valid.notna().sum())
    amount_missing += int(valid.isna().sum())

    if valid.notna().any():

        amount_min = min(
            amount_min,
            valid.min()
        )

        amount_max = max(
            amount_max,
            valid.max()
        )

    if chunk_no % 5 == 0:
        print(
            f"Processed: {total_rows:,} rows"
        )

print("\n" + "=" * 80)
print("RESULT")
print("=" * 80)

print(f"Total rows:       {total_rows:,}")
print(f"Valid amounts:    {amount_nonmissing:,}")
print(f"Missing amounts:  {amount_missing:,}")
print(
    f"Missing %:        "
    f"{amount_missing / total_rows * 100:.6f}%"
)

if amount_nonmissing > 0:
    print(f"Minimum amount:   ${amount_min:,.2f}")
    print(f"Maximum amount:   ${amount_max:,.2f}")
else:
    print("No valid numeric Amount values found.")

FULL IBM AMOUNT VALIDATION - FIXED
Processed: 2,500,000 rows
Processed: 5,000,000 rows
Processed: 7,500,000 rows
Processed: 10,000,000 rows
Processed: 12,500,000 rows
Processed: 15,000,000 rows
Processed: 17,500,000 rows
Processed: 20,000,000 rows
Processed: 22,500,000 rows

RESULT
Total rows:       24,386,900
Valid amounts:    24,386,900
Missing amounts:  0
Missing %:        0.000000%
Minimum amount:   $-500.00
Maximum amount:   $12,390.50
